# Hidroklimatolojik Verilerin Hazırlanması ve Kalite Kontrolü (QA/QC)

Bu çalışma defteri, MGM standart süreli maksimum yağış (`.xlsx`) ile günlük sıcaklık ve toplam yağış (`.txt`) zaman serilerinin **WMO-No. 100 / 1203** standartlarına göre kalite kontrolünü gerçekleştirir.

In [ ]:
import io
import numpy as np
import pandas as pd
from google.colab import files
from IPython.display import display, Markdown

def colab_dosya_yukle(asama_basligi, beklenen_format, ornek_icerik, uzanti):
    display(Markdown(f"### {asama_basligi}"))
    print(f"Beklenen Dosya : {beklenen_format}")
    print(f"Beklenen İçerik:\n{ornek_icerik}\n")
    
    secim = input(f"Bu aşama için '{uzanti}' dosyası yüklemek istiyor musunuz? (e/h): ").strip().lower()
    if secim != 'e':
        print("→ Bu aşama atlandı.\n")
        return None, None
        
    yuklenen = files.upload()
    if not yuklenen:
        return None, None
        
    dosya_adi = list(yuklenen.keys())[0]
    if not dosya_adi.lower().endswith(uzanti):
        print(f"HATA: Beklenen uzantı '{uzanti}', yüklenen '{dosya_adi}'.")
        return None, None
        
    print(f"✓ Yüklendi: {dosya_adi}\n")
    return dosya_adi, io.BytesIO(yuklenen[dosya_adi])

In [ ]:
SURELER = ['5dk', '10dk', '15dk', '30dk', '1sa', '2sa', '3sa', '4sa', '5sa', '6sa', '8sa', '12sa', '18sa', '24sa']

dosya_adi, dosya_icerik = colab_dosya_yukle(
    asama_basligi="1. Aşama: Standart Süreli Maksimum Yağış QA/QC (.xlsx)",
    beklenen_format="MGM Standart Süreli Yağış Tablosu (.xlsx)",
    ornek_icerik="- Sayfa Adı: 'TABLO'\n- Sütunlar: Gözlem Yılı | 5dk | 10dk | ... | 24sa",
    uzanti=".xlsx"
)

if dosya_icerik:
    df_ham = pd.read_excel(dosya_icerik, sheet_name='TABLO', header=None)
    istasyon = str(df_ham.iloc[1, 0]).replace('METEOROLOJİ İSTASYONU', '').strip()
    
    satirlar = []
    for idx in range(5, len(df_ham)):
        try:
            yil = int(float(df_ham.iloc[idx, 0]))
            if 1900 <= yil <= 2030:
                satirlar.append([yil] + list(pd.to_numeric(df_ham.iloc[idx, 1:15], errors='coerce').values))
        except (ValueError, TypeError):
            break
            
    df_standart = pd.DataFrame(satirlar, columns=['Yil'] + SURELER).sort_values('Yil').reset_index(drop=True)
    eksik_oran = ((df_standart['Yil'].max() - df_standart['Yil'].min() + 1 - len(df_standart)) / 
                  (df_standart['Yil'].max() - df_standart['Yil'].min() + 1)) * 100
    
    print(f"İstasyon: {istasyon} | Yıl Sayısı (N): {len(df_standart)} | Eksik Oran: %{eksik_oran:.2f}")
    print("WMO Durumu:", "KABUL" if (len(df_standart) >= 30 and eksik_oran < 10) else "RED")
    display(df_standart.head())

In [ ]:
dosya_adi_txt, txt_icerik = colab_dosya_yukle(
    asama_basligi="2. Aşama: Günlük Yağış / Sıcaklık Zaman Serisi QA/QC (.txt)",
    beklenen_format="MGM Günlük Gözlem Metin Dosyası (.txt)",
    ornek_icerik="- Ayraç: '|' (Kodlama: cp1254)\n- Sütunlar: İstasyon No | Yıl | Ay | Gün | Değer",
    uzanti=".txt"
)

if txt_icerik:
    param_tipi = input("Yüklenen parametre türü nedir? ('yagis' veya 'sicaklik'): ").strip().lower()
    df_gunluk = pd.read_csv(txt_icerik, sep="|", encoding="cp1254")
    df_gunluk.columns = [c.strip() for c in df_gunluk.columns]
    df_gunluk['Deger'] = pd.to_numeric(df_gunluk.iloc[:, -1], errors='coerce')
    
    if param_tipi == 'yagis':
        df_gunluk.loc[(df_gunluk['Deger'] < 0) | (df_gunluk['Deger'] > 500), 'Deger'] = np.nan
    else:
        df_gunluk.loc[(df_gunluk['Deger'] < -45) | (df_gunluk['Deger'] > 55), 'Deger'] = np.nan
        
    aylik_eksik = df_gunluk.groupby(['Yıl', 'Ay'])['Deger'].apply(lambda x: x.isna().sum())
    for (y, m) in aylik_eksik[aylik_eksik > 5].index:
        df_gunluk.loc[(df_gunluk['Yıl'] == y) & (df_gunluk['Ay'] == m), 'Deger'] = np.nan
        
    gecerli_yillar = df_gunluk.groupby('Yıl')['Deger'].count()
    gecerli_yillar = gecerli_yillar[gecerli_yillar >= 330].index.tolist()
    df_temiz = df_gunluk[df_gunluk['Yıl'].isin(gecerli_yillar)].copy()
    print(f"WMO Geçerli Yıl Sayısı: {len(gecerli_yillar)}")
    display(df_temiz.head())

display(Markdown("---\n**[← Dokümantasyon Sayfasına Geri Dön (mkoztas.github.io)](https://mkoztas.github.io/calismalar/verihazirlik/)**"))